In [12]:
import pandas as pd
import re


fin_2012_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Small Town:City Economic Data/2012_Individual_Unit_file (1)/2012FinEstDAT_10162019modp_pu.txt"

colspecs = [(0, 14), (14, 17), (17, 29), (29, 33), (33, 34)]
colnames = ['ID', 'item_code', 'amount', 'year', 'flag']

df = pd.read_fwf(fin_2012_path, colspecs=colspecs, header=None, names=colnames, dtype=str)


df['state_fips'] = df['ID'].str[0:2]
df['gov_type']   = df['ID'].str[2]
df['county_fips'] = df['ID'].str[3:6]
df['unit_id']    = df['ID'].str[6:9]

df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

df = df[df['gov_type'].isin(['2', '3'])].reset_index(drop=True)

In [13]:
GID_PATH = "/Users/mayarabin/Desktop/Thesis Downloads V2/Small Town:City Economic Data/2012_Individual_Unit_file (1)/Fin_GID_2012.txt"

gid_colspecs = [
    (0,   14),   # ID code
    (14,  78),   # ID name
    (78,  113),  # County name
    (113, 115),  # FIPS state code
    (115, 118),  # FIPS county code
    (118, 123),  # FIPS place code
    (123, 132),  # Population
    (132, 134),  # Population year
    (134, 141),  # Enrollment
    (141, 143),  # Enrollment year
    (143, 145),  # Function code
    (145, 147),  # School level code
    (147, 151),  # Fiscal year ending
    (151, 153),  # Survey year
]

gid_colnames = [
    'ID', 'id_name', 'county_name', 'fips_state', 'fips_county',
    'fips_place', 'population', 'pop_year', 'enrollment',
    'enrollment_year', 'function_code', 'school_level',
    'fiscal_year_end', 'survey_year'
]

gid = pd.read_fwf(GID_PATH, colspecs=gid_colspecs, header=None, names=gid_colnames, dtype=str)

In [14]:
df_merged = df.merge(gid[['ID', 'id_name', 'county_name', 'fips_state', 'fips_county', 'fips_place', 'population']], 
                     on='ID', 
                     how='left')


In [15]:
KEEP_CODES = [
    'B01', 'B21', 'B22', 'B30', 'B42', 'B46', 'B50', 'B59', 'B79', 'B80', 'B89',
    'T01', 'T09', 'T10', 'T11', 'T13', 'T16', 'T19', 'T20', 'T21', 'T22', 'T27', 'T28', 'T29'
]

df_merged = df_merged[df_merged['item_code'].isin(KEEP_CODES)].reset_index(drop=True)



In [16]:
# Split into B and T totals
df_agg = (df_merged.groupby(['ID', 'id_name', 'county_name', 'fips_state', 'fips_county', 'fips_place', 'population'])
          .apply(lambda x: pd.Series({
              'federal_transfers_raw': x.loc[x['item_code'].str.startswith('B'), 'amount'].sum(),
              'tax_revenue_raw':       x.loc[x['item_code'].str.startswith('T'), 'amount'].sum()
          }))
          .reset_index())



In [17]:
deflator_2012 = 93.460
deflator_2017 = 100.000

df_agg['federal_transfers_real'] = df_agg['federal_transfers_raw'] * (deflator_2017 / deflator_2012)
df_agg['tax_revenue_real']       = df_agg['tax_revenue_raw']       * (deflator_2017 / deflator_2012)

In [18]:
PLACES_PATH = "/Users/mayarabin/Desktop/Thesis Files/places_panel_final.parquet"

# Load places panel
places = pd.read_parquet(PLACES_PATH)

In [19]:

_SUFFIXES = re.compile(
    r'\b(city|town|village|borough|township|municipality|cdp|'
    r'plantation|grant|location|purchase|gore)\b',
    re.IGNORECASE,
)

def _base(name) -> str:
    if not isinstance(name, str):
        return ''
    name = name.lower()
    name = re.sub(r"[''`\-]", ' ', name)
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return re.sub(r'\s+', ' ', name).strip()


def normalize_with_suffix(name: str) -> str:
    return _base(name)

def normalize_stripped(name: str) -> str:
    return _SUFFIXES.sub('', _base(name)).strip()

In [24]:
MCD_STATES = {
    'Connecticut', 'Delaware', 'Indiana', 'Kansas', 'Maine', 'Maryland',
    'Massachusetts', 'Minnesota', 'New Hampshire', 'New Jersey', 'New York',
    'Pennsylvania', 'Rhode Island', 'Vermont', 'Virginia', 'Wisconsin'
}

# Build fips_state -> state_name mapping from places panel
fips_to_state = (places[['PLACE_ID', 'state_name']]
                 .assign(fips=places['PLACE_ID'].astype(str).str[:2])
                 .drop_duplicates(subset='fips')
                 .set_index('fips')['state_name']
                 .to_dict())

# Construct PLACE_ID in financial data based on MCD state or not
def build_geoid(row):
    state = fips_to_state.get(row['fips_state'], '')
    if state in MCD_STATES:
        return row['fips_state'] + row['fips_county'] + row['fips_place']
    else:
        return row['fips_state'] + row['fips_place']

df_agg['PLACE_ID'] = df_agg.apply(build_geoid, axis=1)

# Filter places panel to 2010 decade
places_2010 = places[places['year'] == 2010].copy()
places_2010['PLACE_ID'] = places_2010['PLACE_ID'].astype(str).str.strip()

print(f"Places panel 2010 slice: {len(places_2010):,} rows")
print(f"Financial GEOIDs — 7-digit: {(df_agg['PLACE_ID'].str.len() == 7).sum():,}")
print(f"Financial GEOIDs — 10-digit: {(df_agg['PLACE_ID'].str.len() == 10).sum():,}")

# Pass 1: merge on PLACE_ID
merged = df_agg.merge(places_2010, on='PLACE_ID', how='left')

print(f"\nAfter PLACE_ID merge — matched: {merged['state_name'].notna().sum():,}, "
      f"unmatched: {merged['state_name'].isna().sum():,}")

# Build name lookups from places_2010
lookup_name_with     = {}
lookup_name_stripped = {}

for _, row in places_2010.iterrows():
    place_id = str(row['PLACE_ID']).strip()
    state    = row['state_name']
    lookup_name_with.setdefault((state, normalize_with_suffix(row['NAME'])), []).append(place_id)
    lookup_name_stripped.setdefault((state, normalize_stripped(row['NAME'])), []).append(place_id)

# Pass 2: name matching on unmatched rows
new_place_ids = {}
counts = {'matched_with': 0, 'matched_stripped': 0, 'unmatched': 0}

for idx in merged[merged['state_name'].isna()].index:
    fips     = merged.at[idx, 'fips_state']
    state    = fips_to_state.get(fips, '')
    raw_name = merged.at[idx, 'id_name']

    if not state or not isinstance(raw_name, str):
        counts['unmatched'] += 1
        continue

    candidates = lookup_name_with.get((state, normalize_with_suffix(raw_name)), [])
    if len(candidates) == 1:
        new_place_ids[idx] = candidates[0]
        counts['matched_with'] += 1
        continue
    if len(candidates) > 1:
        counts['unmatched'] += 1
        continue

    candidates = lookup_name_stripped.get((state, normalize_stripped(raw_name)), [])
    if len(candidates) == 1:
        new_place_ids[idx] = candidates[0]
        counts['matched_stripped'] += 1
    else:
        counts['unmatched'] += 1

# Update PLACE_IDs and re-merge places panel columns for newly matched rows
for idx, place_id in new_place_ids.items():
    merged.at[idx, 'PLACE_ID'] = place_id

places_cols = [c for c in places_2010.columns if c != 'PLACE_ID']
merged = merged.drop(columns=places_cols).merge(places_2010, on='PLACE_ID', how='left')

Places panel 2010 slice: 30,724 rows
Financial GEOIDs — 7-digit: 20,748
Financial GEOIDs — 10-digit: 14,472

After PLACE_ID merge — matched: 26,758, unmatched: 8,462


In [25]:
df_agg['population'] = pd.to_numeric(df_agg['population'], errors='coerce')

df_agg['federal_transfers_pc'] = (df_agg['federal_transfers_real'] * 1000) / df_agg['population']
df_agg['tax_revenue_pc']       = (df_agg['tax_revenue_real'] * 1000)       / df_agg['population']


# Replace inf (division by zero population) with NaN
df_agg['federal_transfers_pc'] = df_agg['federal_transfers_pc'].replace([float('inf'), float('-inf')], float('nan'))
df_agg['tax_revenue_pc']       = df_agg['tax_revenue_pc'].replace([float('inf'), float('-inf')], float('nan'))

print(df_agg[['id_name', 'population', 'federal_transfers_raw', 'federal_transfers_pc', 'tax_revenue_raw', 'tax_revenue_pc']].head(10).to_string())

             id_name  population  federal_transfers_raw  federal_transfers_pc  tax_revenue_raw  tax_revenue_pc
0  AUTAUGAVILLE TOWN         879                      0              0.000000               40       48.690624
1   BILLINGSLEY TOWN         144                      0              0.000000               65      482.975486
2    PRATTVILLE CITY       34683                      0              0.000000            26452      816.048708
3   BAY MINETTE CITY        8639                      0              0.000000             7231      895.589743
4        DAPHNE CITY       22844                   1067             49.976575            23047     1079.484656
5       ELBERTA TOWN        1574                      0              0.000000             1430      972.087890
6      FAIRHOPE CITY       16481                    212             13.763425            12366      802.823185
7         FOLEY CITY       15345                      0              0.000000            16654     1161.250438
8

In [26]:
unmatched = merged[merged['state_name'].isna()].copy()
unmatched['state_name_derived'] = unmatched['fips_state'].map(fips_to_state)

unmatched.to_csv("/Users/mayarabin/Desktop/Thesis Downloads V2/Financial_2012_unmatched_final.csv", index=False)
print(f"Saved {len(unmatched):,} unmatched rows.")
print(unmatched.groupby('state_name_derived')['ID'].count().sort_values(ascending=False).to_string())

Saved 5,733 unmatched rows.
state_name_derived
North Dakota      1112
Illinois           800
Ohio               749
South Dakota       632
Indiana            517
Kansas             351
New York           339
Nebraska           278
Missouri           177
Michigan           164
Virginia           158
Maryland            61
Delaware            50
Vermont             15
Utah                 9
Minnesota            9
Connecticut          6
Wisconsin            5
Maine                4
Kentucky             4
Pennsylvania         4
Georgia              4
Alabama              4
Iowa                 3
Texas                3
Arkansas             3
Tennessee            2
Florida              2
New Jersey           1
Arizona              1
North Carolina       1
Oklahoma             1
Oregon               1
South Carolina       1
